# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List
from time import time, sleep

In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Client` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Client:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Client(api_key=k)

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods (payment client)

In [ ]:
#| export
@patch
def _request(self: Client, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, **kwargs)

In [ ]:
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

Let's use a helper function to process the response.

In [ ]:
#| export
def _process_response(r):
    "Process response: raise for status and return json if possible"
    r.raise_for_status()
    try: return r.json()
    except: return r

### User Info

In [ ]:
#| export

@patch
def me(self: Client):
    "Retrieve the user's info."
    r = self._request("GET", "v0/users/me")
    return _process_response(r)

In [ ]:
fs.me()

{'name': 'Fewsats',
 'last_name': 'Tester',
 'email': 'test@fewsats.com',
 'billing_info': None,
 'id': 15,
 'created_at': '2024-12-18T18:19:00.531Z'}

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Client):
    "Retrieve the balance of the user's wallet."
    r = self._request("GET", "v0/wallets")
    return _process_response(r)

In [ ]:
fs.balance()

[{'id': 15, 'balance': 4958, 'currency': 'usd'}]

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Client) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    r = self._request("GET", "v0/stripe/payment-methods")
    return _process_response(r)

In [ ]:
pm = fs.payment_methods()
pm

[{'id': 5,
  'last4': '4242',
  'brand': 'Visa',
  'exp_month': 12,
  'exp_year': 2034,
  'is_default': True}]

In [ ]:
assert isinstance(pm, list)

### Preview a Purchase

Preview the resulting state of a purchase. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def _preview_payment(self: Client,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return _process_response(self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount}))


In [ ]:
p = fs._preview_payment(amount="300") # 3.00 USD
p

{'invoice': {'description': 'USD amount preview',
  'amount_usd': 300,
  'amount_btc': 0,
  'macaroon': '',
  'invoice': ''},
 'transaction': {'current_balance': 4958,
  'balance_to_apply': 300,
  'amount_to_charge': 0,
  'final_balance': 4658},
 'already_purchased': False,
 'purchase': None}

The following is an example of how to pay a lightning invoice. This is a low level method that should not used by most users. This method will use the default payment method if a charge is needed.

### Pay 

The pay method is asynchronous. It returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.

In [ ]:
#| export
@patch
def _submit_payment(self:Client,
        purl:str, # payment endpoint URL
        pct:str, # payment context token
        # offer fields
        amount:int, # amount in cents
        currency:str, # currency
        description:str, # description
        offer_id:str, # offer id
        payment_methods:list[str], # payment methods
        title:str, # offer title
        type:str, # offer type
        balance:int = 0, # balance (optional)
        pm:str = '', # preferred payment method (optional)
) -> dict: # payment status response
    "POST payment request. Returns payment status response"
    return _process_response(self._request("POST", "v0/l402/purchases/from-offer", json={
        "payment_request_url": purl,
        "payment_context_token": pct,
        "payment_method": pm,
        "offer": {
            "offer_id": offer_id,
            "title": title,
            "description": description,
            "amount": amount,
            "type": type,
            "currency": currency,
            "balance": balance,
            "payment_methods": payment_methods,
        },
    }))

In [ ]:
# Example offer from demo replit: https://replit.com/t/fewsats/repls/PaymentOfferServer/view
ofs = httpx.get("https://l402-offers.replit.app").json()
ofs

{'offers': [{'offer_id': '4e83e2fb-c9e1-4c29-af1a-2407ea5f8ddd',
   'amount': 1,
   'currency': 'USD',
   'description': 'Purchase 1 credit for API access',
   'title': '1 Credit Package',
   'type': 'one-time',
   'payment_methods': ['onchain']}],
 'payment_context_token': '5001a8f1-2318-404d-882a-1d92c55673a6',
 'payment_request_url': 'https://hub-5n97k.ondigitalocean.app/v0/l402/payment-request',
 'version': '0.2.2'}

The pay method allows to specify which payment method to use. If not specified, the backend will decide which payment to use.

In [ ]:
r = fs._submit_payment(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][-1])
r

HTTPStatusError: Server error '500 Internal Server Error' for url 'https://hub-5n97k.ondigitalocean.app/v0/l402/purchases/from-offer'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

Lightning payments have almost instant settlement so often the status will be `success` right away. For credit card payments, we'll have to wait for the stripe payment to settle.

In [ ]:
r = fs._submit_payment(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][-1], pm='credit_card')
r

{'id': 266,
 'created_at': '2025-01-26T01:48:57.689Z',
 'status': 'pending',
 'payment_method': 'credit_card'}

### Payment info 

We can check all the details as follows:

In [ ]:
#| export
@patch
def payment_info(self:Client,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return _process_response(self._request("GET", f"v0/l402/purchases/{pid}"))

In [ ]:
pid = r['id']
r = fs.payment_info(pid)
r

{'id': 266,
 'created_at': '2025-01-26T01:48:57.689Z',
 'status': 'pending',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'invoice': '',
 'preimage': '',
 'amount': 499,
 'currency': 'usd',
 'payment_method': 'credit_card',
 'title': '750 Credits Package',
 'description': 'Purchase 750 credits for API access',
 'type': 'top-up'}

After the stripe payment settles, the status will be updated to `success`.

In [ ]:
#| skip
r = fs.payment_info(pid)
r['status']

'pending'

### Wait for settlement

For convnience we can use the method `_wait_for_settlement` to wait for the payment to settle.

In [ ]:
#| export
@patch
def _wait_for_settlement(self:Client,
                        pid:str, # purchase id
                        max_interval:int=120, # maximum interval between checks in seconds
                        max_wait:int=600): # maximum total wait time in seconds
    "Wait for payment settlement with exponential backoff"
    start,wait = time(),1
    while time() - start < max_wait:
        r = self.payment_info(pid)
        status = r['status']
        if status == 'success': return r
        if status == 'failed': raise ValueError(f"Payment {pid} failed")
        sleep(min(wait, max_interval))
        wait *= 2
    raise TimeoutError(f"Payment {pid} did not settle within {max_wait} seconds. Final status: {status}")

In [ ]:
#| skip
r = fs._wait_for_settlement('223') # test payment already settled
r

{'id': 223,
 'created_at': '2024-12-26T10:01:00.233Z',
 'status': 'success',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'invoice': '',
 'preimage': '',
 'amount': 499,
 'currency': 'usd',
 'payment_method': 'credit_card',
 'title': '750 Credits Package',
 'description': 'Purchase 750 credits for API access',
 'type': 'top-up'}

### Pay

The `pay` method submits and payment request and waits for settlement.


In [ ]:
#| export
@patch
def pay(self:Client,
        purl:str, # payment endpoint URL
        pct:str, # payment context token
        # offer fields
        amount:int, # amount in cents
        balance:int, # balance
        currency:str, # currency
        description:str, # description
        offer_id:str, # offer id
        payment_methods:list[str], # payment methods
        title:str, # offer title
        type:str, # offer type
        pm:str = '', # preferred payment method (optional)
) -> dict: # payment status response
    "Pay for an offer and wait for settlement"
    r = self._submit_payment(purl, pct, amount, balance, currency, description, offer_id, payment_methods, title, type, pm)
    return self._wait_for_settlement(r['id'])


In [ ]:
r = fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][0], pm='lightning')
r

{'id': 267,
 'created_at': '2025-01-26T01:49:01.352Z',
 'status': 'success',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'invoice': 'lnbc90n1pnet9yvpp5vxl39upgtzkz3m0awmkcw4j56jcagtvs5snvcgluwvvzr29vt6dsdq6xysyxun9v35hggzsv93kkct8v5cqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp58dmfyrjaj5gs5ef83px3xz9fvzd7uekjlrraf4sjlej8c5ty8jrs9qxpqysgqyy3r8mkdyq474xmtn4dz6yfxwu3xdcy4frlze3whdp0nztv68l8yx683sy37kp7y24ms0y4l0ph7myegtl6lqwxxu688ycv9jvqjrpgq6ffya7',
 'preimage': '7891a5148355739e0daca266c6e24a7fc1aa683be1d7a3eb268e0409d0a71e27',
 'amount': 1,
 'currency': 'usd',
 'payment_method': 'lightning',
 'title': '1 Credit Package',
 'description': 'Purchase 1 credit for API access',
 'type': 'top-up'}

In [ ]:
#| export

@patch
def pay_lightning(self: Client, 
                  invoice: str, # lightning invoice
                  description: str = "" ): # description of the payment 
    "Pay for a lightning invoice"
    data = {
        "invoice": invoice,
        "description": description
    }
    return self._request("POST", "v0/l402/purchases/lightning", json=data)

## Methods (payment server)

### Create Offers

In [ ]:
#| export
@patch
def create_offers(self:Client,
                 offers:List[Dict[str,Any]], # List of offer objects following OfferCreateV0 schema
) -> dict:
    "Create offers for L402 payment server"
    return _process_response(self._request("POST", "v0/l402/offers", json={"offers": offers}))

In [ ]:
fs = Client()
# fs = Client(api_key=os.getenv("FEWSATS_LOCAL_API_KEY"), base_url="http://localhost:8000")
test_offers = [{
    "offer_id": "test_offer_2",
    "amount": 100,
    "currency": "usd",
    "description": "Test offer",
    "title": "Test Package",
    "type": "top-up",
    "payment_methods": ["lightning", "credit_card"]
}]

r = fs.create_offers(test_offers)
r

{'offers': [{'offer_id': 'test_offer_2',
   'amount': 100,
   'currency': 'usd',
   'description': 'Test offer',
   'title': 'Test Package',
   'type': 'top-up',
   'payment_methods': ['lightning', 'credit_card']}],
 'payment_context_token': 'e9c062f5-f9d5-4ae1-8905-6b49ae093826',
 'payment_request_url': 'https://hub-5n97k.ondigitalocean.app/v0/l402/payment-request',
 'version': '0.2.2'}

## As tools

In [ ]:
#| export

@patch
def as_tools(self:Client):
    "Return list of available tools for AI agents"
    return [
        self.me,
        self.balance,
        self.payment_methods,
        self.pay,
        self.payment_info,
    ]

In [ ]:
fs.as_tools()

[<bound method Client.me of <__main__.Client object>>,
 <bound method Client.balance of <__main__.Client object>>,
 <bound method Client.payment_methods of <__main__.Client object>>,
 <bound method Client.pay of <__main__.Client object>>,
 <bound method Client.payment_info of <__main__.Client object>>]

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance()

[{'id': 15, 'balance': 4959, 'currency': 'usd'}]

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer using lightning {ofs}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01RCWZnqoZXUsQJgWpUwqZ7Z', content=[TextBlock(text='Certainly! I\'d be happy to help you pay for the cheapest offer using Lightning. Let\'s analyze the offers and proceed with the payment.\n\nFrom the information you\'ve provided, the cheapest offer is:\n\n- Amount: 1 cent (USD)\n- Balance: 1 credit\n- Offer ID: offer_c668e0c0\n- Title: "1 Credit Package"\n- Type: top-up\n- Payment method: Lightning\n\nNow, let\'s use the `pay` function to process this payment. We\'ll need to use the information from the cheapest offer and the payment context you\'ve provided.', type='text'), ToolUseBlock(id='toolu_01MCXoRCLnskTnYTEwHDVQyw', input={'purl': 'https://stock.l402.org/l402/payment-request', 'pct': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, name='pay', type='tool_use')], mo

Great news! The payment for the cheapest offer has been successfully processed. Here's a summary of the transaction:

1. Payment Status: Success
2. Amount Paid: 1 cent (USD)
3. Payment Method: Lightning
4. Title: 1 Credit Package
5. Description: Purchase 1 credit for API access
6. Type: Top-up
7. Transaction ID: 264
8. Created At: 2025-01-25T19:52:55.228Z (Note: This appears to be a future date, which might be due to a system time discrepancy)

The payment has been completed, and you should now have 1 credit added to your API access. Is there anything else you would like me to help you with regarding this transaction or any other matters?

<details>

- id: `msg_016QxjwdSQchw5Vmyqqu23ey`
- content: `[{'text': "Great news! The payment for the cheapest offer has been successfully processed. Here's a summary of the transaction:\n\n1. Payment Status: Success\n2. Amount Paid: 1 cent (USD)\n3. Payment Method: Lightning\n4. Title: 1 Credit Package\n5. Description: Purchase 1 credit for API access\n6. Type: Top-up\n7. Transaction ID: 264\n8. Created At: 2025-01-25T19:52:55.228Z (Note: This appears to be a future date, which might be due to a system time discrepancy)\n\nThe payment has been completed, and you should now have 1 credit added to your API access. Is there anything else you would like me to help you with regarding this transaction or any other matters?", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 2146, 'output_tokens': 181}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [ ]:
fs.balance(), chat.h

([{'id': 15, 'balance': 4958, 'currency': 'usd'}],
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer using lightning {'offers': [{'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, {'amount': 100, 'balance': 120, 'currency': 'USD', 'description': 'Purchase 120 credits for API access', 'offer_id': 'offer_97bf23f7', 'payment_methods': ['lightning', 'coinbase_commerce'], 'title': '120 Credits Package', 'type': 'top-up'}, {'amount': 499, 'balance': 750, 'currency': 'USD', 'description': 'Purchase 750 credits for API access', 'offer_id': 'offer_a896b13c', 'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'], 'title': '750 Credits Package', 'type': 'top-up'}], 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'payment_request_url': 'https://stock.l402.org/l40

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()